# Verbatim Quran Whisper fine-tune (Colab, T4 GPU)
Teaches `tarteel-ai/whisper-base-ar-quran` to write what was **actually** recited, including omissions,
repetitions, substitutions and early stops, instead of silently repairing them.

Training audio is spliced from professional recitations using quran.com word timings, with noise/gain
augmentation toward phone audio. Held-out reciters measure the **repair rate** before and after.

Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Outputs go to Google Drive.

In [ ]:
!pip -q install "transformers>=4.44" datasets evaluate jiwer librosa soundfile accelerate requests
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, random, re, requests, numpy as np, librosa, soundfile as sf, torch
from tqdm.auto import tqdm
from pathlib import Path

BASE = 'tarteel-ai/whisper-base-ar-quran'
OUT = Path('/content/drive/MyDrive/quran-verbatim'); OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); (WORK / 'clips').mkdir(parents=True, exist_ok=True)
TRAIN_RECITERS = [1, 3, 4, 5, 6, 7, 9, 10, 161, 174]   # quran.com ids with word timings
EVAL_RECITERS = [2, 168]                                 # held out: never trained on
SURAHS = list(range(1, 115))
SURAHS_PER_RECITER = 25    # raise for more data; 25 keeps download + generation under ~1 h
MAX_SECONDS = 28
random.seed(0); np.random.seed(0)

def normalize(text):
    # Same rules as on-device/Native/ArabicMatching.swift.
    for pattern, repl in [(r'[\u064B-\u065F\u0670\u06D6-\u06ED\u0640]', ''), (r'[\u0622\u0623\u0625\u0671]', 'ا'),
                          (r'\u0649', 'ي'), (r'\u0629', 'ه'), (r'\u0624', 'و'), (r'\u0626', 'ي'),
                          (r'[^\u0621-\u064A\s]', '')]:
        text = re.sub(pattern, repl, text)
    return text.split()

In [ ]:
API = 'https://api.quran.com/api/v4'
def surah_words(surah):
    words, page = {}, 1
    while page:
        r = requests.get(f'{API}/verses/by_chapter/{surah}', params={'words': 'true', 'word_fields': 'text_imlaei',
                         'per_page': 50, 'page': page}, timeout=30).json()
        for v in r['verses']:
            words[v['verse_key']] = [w['text_imlaei'] for w in v['words'] if w['char_type_name'] == 'word']
        page = r['pagination']['next_page']
    return words

def surah_audio(reciter, surah):
    meta = requests.get(f'{API}/chapter_recitations/{reciter}/{surah}', params={'segments': 'true'}, timeout=30).json()['audio_file']
    mp3 = WORK / f'{reciter}_{surah}.mp3'
    if not mp3.exists(): mp3.write_bytes(requests.get(meta['audio_url'], timeout=120).content)
    audio, _ = librosa.load(mp3, sr=16000); mp3.unlink()
    return audio, {t['verse_key']: [s for s in t['segments'] if len(s) == 3 and s[2] > s[1]] for t in meta['timestamps']}

TEXT = {}
def text_for(surah):
    if surah not in TEXT: TEXT[surah] = surah_words(surah)
    return TEXT[surah]

In [ ]:
def piece(audio, seg):
    a, b = int(seg[1] * 16), int(seg[2] * 16)
    return audio[a:b]

def splice(pieces):
    gap = np.zeros(int(16000 * random.uniform(0.03, 0.25)), np.float32)
    out = []
    for p in pieces: out += [p, gap]
    return np.concatenate(out)

def augment(x):
    x = x * random.uniform(0.3, 1.2)
    if random.random() < 0.7:  # room / phone noise at 10-35 dB SNR
        snr = random.uniform(10, 35)
        x = x + np.random.randn(len(x)).astype(np.float32) * np.sqrt(np.mean(x ** 2) / 10 ** (snr / 10))
    return np.clip(x, -1, 1)

def examples_for_ayah(audio, segs, words, other_pool):
    # Returns (audio, said_words, kind). said_words is the verbatim target.
    segs = [s for s in segs if 1 <= s[0] <= len(words)]
    if len(segs) < 3: return []
    idx = [s[0] - 1 for s in segs]
    pieces = [piece(audio, s) for s in segs]
    out = [(splice(pieces), [words[i] for i in idx], 'clean')]
    k = random.randrange(1, len(segs) - 1)
    out.append((splice(pieces[:k] + pieces[k + 1:]), [words[i] for i in idx[:k] + idx[k + 1:]], 'omission'))
    k = random.randrange(0, len(segs) - 1)
    out.append((splice(pieces[:k + 1] + pieces[k:]), [words[i] for i in idx[:k + 1] + idx[k:]], 'repetition'))
    if other_pool:
        k = random.randrange(1, len(segs))
        sub_audio, sub_word = random.choice(other_pool)
        if normalize(sub_word) != normalize(words[idx[k]]):
            said = [words[i] for i in idx]; said[k] = sub_word
            out.append((splice(pieces[:k] + [sub_audio] + pieces[k + 1:]), said, 'substitution'))
    k = random.randrange(1, len(segs) - 1)
    out.append((splice(pieces[:k]), [words[i] for i in idx[:k]], 'unfinished'))
    return [(augment(a), w, kind) for a, w, kind in out if len(a) <= MAX_SECONDS * 16000]

def build(reciters, name, surahs_per_reciter):
    rows = []
    for reciter in reciters:
        for surah in tqdm(random.sample(SURAHS, surahs_per_reciter), desc=f'{name} reciter {reciter}'):
            try: audio, timing = surah_audio(reciter, surah)
            except Exception as e: print('skip', reciter, surah, e); continue
            text = text_for(surah)
            pool = [(piece(audio, s), text[k][s[0] - 1]) for k, ss in timing.items() if k in text
                    for s in ss if 1 <= s[0] <= len(text[k])]
            for key, segs in timing.items():
                if key not in text: continue
                for n, (clip, said, kind) in enumerate(examples_for_ayah(audio, segs, text[key], pool)):
                    path = WORK / 'clips' / f'{name}_{reciter}_{key.replace(":", "_")}_{n}.wav'
                    sf.write(path, clip, 16000, subtype='PCM_16')
                    rows.append({'path': str(path), 'text': ' '.join(said), 'kind': kind,
                                 'correct': ' '.join(text[key]), 'reciter': reciter})
        print(name, reciter, len(rows))
    (WORK / f'{name}.json').write_text(json.dumps(rows, ensure_ascii=False))
    return rows

train_rows = build(TRAIN_RECITERS, 'train', SURAHS_PER_RECITER)
eval_rows = random.sample(build(EVAL_RECITERS, 'eval', 6), 600)
print(len(train_rows), 'train', len(eval_rows), 'eval')

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained(BASE, language='arabic', task='transcribe')
device = 'cuda'

from transformers import GenerationConfig

def transcribe(model, rows, batch=16):
    # Tarteel's checkpoint ships an old generation config without language ids; borrow whisper-base's (same tokenizer).
    if not hasattr(model.generation_config, 'lang_to_id'):
        model.generation_config = GenerationConfig.from_pretrained('openai/whisper-base')
    model.eval(); outs = []
    for i in tqdm(range(0, len(rows), batch), desc='transcribing', unit='batch'):
        audio = [sf.read(r['path'])[0].astype(np.float32) for r in rows[i:i + batch]]
        feats = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features
        with torch.no_grad():
            ids = model.generate(feats.to(device, torch.float16 if model.dtype == torch.float16 else torch.float32),
                                 language='ar', task='transcribe', max_new_tokens=200)
        outs += processor.batch_decode(ids, skip_special_tokens=True)
    return outs

def report(rows, preds, title):
    import jiwer
    stats = {}
    for r, p in zip(rows, preds):
        s = stats.setdefault(r['kind'], {'n': 0, 'exact': 0, 'repaired': 0})
        s['n'] += 1
        s['exact'] += normalize(p) == normalize(r['text'])
        s['repaired'] += r['kind'] != 'clean' and normalize(p) == normalize(r['correct'])
    wer = jiwer.wer([' '.join(normalize(r['text'])) for r in rows], [' '.join(normalize(p)) or '-' for p in preds])
    print(title, f'WER {wer:.3f}')
    for k, s in sorted(stats.items()):
        print(f"  {k:<13} n={s['n']:<4} verbatim={s['exact'] / s['n']:.2f}  repaired={s['repaired'] / s['n']:.2f}")
    return wer

baseline = WhisperForConditionalGeneration.from_pretrained(BASE).to(device)
report(eval_rows, transcribe(baseline, eval_rows), 'BASELINE (held-out reciters)')
del baseline; torch.cuda.empty_cache()

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

model = WhisperForConditionalGeneration.from_pretrained(BASE)
model.generation_config.language = 'ar'; model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

@dataclass
class Collator:
    def __call__(self, rows):
        audio = [sf.read(r['path'])[0].astype(np.float32) for r in rows]
        feats = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features
        labels = processor.tokenizer([r['text'] for r in rows], return_tensors='pt', padding=True)
        ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == model.config.decoder_start_token_id).all(): ids = ids[:, 1:]
        return {'input_features': feats, 'labels': ids}

args = Seq2SeqTrainingArguments(
    output_dir=str(OUT / 'ckpt'), per_device_train_batch_size=16, gradient_accumulation_steps=1,
    learning_rate=1e-5, warmup_steps=300, max_steps=4000, fp16=True, logging_steps=50, disable_tqdm=False, logging_first_step=True,
    save_steps=1000, save_total_limit=2, report_to='none', remove_unused_columns=False,
    dataloader_num_workers=2)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Dataset.from_list(train_rows),
                         data_collator=Collator())
trainer.train(resume_from_checkpoint=any((OUT / 'ckpt').glob('checkpoint-*')))  # survives Colab disconnects
model.save_pretrained(OUT / 'hf'); processor.save_pretrained(OUT / 'hf')

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(OUT / 'hf').to(device)
report(eval_rows, transcribe(model, eval_rows), 'FINE-TUNED (held-out reciters)')

Export for the phone: whisper.cpp GGML F16, same format as the bundled model.
Download `ggml-model.bin` from Drive → `quran-verbatim/` and give it to Claude for the local 70-clip evaluation before it replaces the app model.

In [ ]:
%cd /content
!git clone -q https://github.com/ggml-org/whisper.cpp && cd whisper.cpp && git checkout -q d1be6fde11ac6e0407606b4e42fe72d34add8037
!git clone -q --depth 1 https://github.com/openai/whisper
!python whisper.cpp/models/convert-h5-to-ggml.py {OUT}/hf whisper {OUT}
!ls -la {OUT}; sha256sum {OUT}/ggml-model.bin